# Entra Agent ID: watch an agent act on behalf of a user

**The idea:** the user brings a ticket, the agent brings a badge, and Entra exchanges both for a pass to Microsoft Graph.

1. **Tc -- the user's ticket.** A normal client app signs the user in and gets an **access token for the blueprint's API**, not for Graph.
2. **T1 -- the agent's badge.** The blueprint authenticates with its own credential and names its child agent identity using `fmi_path`.
3. **Tg -- the Graph pass.** The child agent presents **T1 as its credential** and **Tc as the user assertion**. Entra issues a new token for Graph, subject to consent and policy.
4. **Use the pass.** Graph allows reading that user's profile, but not listing the directory's users.

All three tokens come from **Microsoft Entra ID**. The client and agent do not mint them. The notebook plays the frontend and backend in one kernel; there is no LLM or deployed agent service here, but the **agent identity and token exchanges are real**.

![Sequence drawing showing the user, client app, blueprint, child agent, Entra ID, and Microsoft Graph exchanging Tc, T1, and Tg.](entra-obo-token-exchange.svg)

Read from top to bottom. Setup and consent happen beforehand; the blueprint and child agent are two identities used by the same backend, not necessarily separate services.

| Name | What it is | What to watch |
|------|------------|---------------|
| Blueprint | Agent template, API audience, and credential holder | `BLUEPRINT_APP_ID` |
| Agent identity | One child identity created from the blueprint | `AGENT_APP_ID` |
| Public client | Ordinary app that signs the user in | `CLIENT_APP_ID` |

## Before the audience arrives

Follow **Setup** and **Permissions** in `README.md`. Use a development tenant, Python 3.11+, and the `.venv` kernel. Set `TENANT_ID` in `.env`; choose a unique `AGENT_PREFIX`.

Run **sections 1-6 beforehand**. They provision objects and grant only `User.Read` to **one demo user**. The default is the same account as the provisioning admin; set `DEMO_USER_OBJECT_ID` to use a different member user. Then present **sections 7-11** live. Do not restart the kernel between these sections.

The admin needs **Agent ID Administrator** for this secret-based setup and **Cloud Application Administrator** (or another approved role) for the ordinary client and delegated consent. **Agent ID Developer alone is insufficient for adding a secret.** Roles and Graph OAuth scopes are separate requirements; the full scope list is in section 2 and the README.

> Token inspection is educational, not signature validation. Graph access tokens can be opaque: the demo reports that explicitly and still calls Graph. A production backend must validate incoming Tc with a supported library before using it. Never paste tokens into online decoders.
>
> A private local state file holds the blueprint secret in plaintext for this development demo. Its default location is **outside the repository and OneDrive**. Clear all outputs before sharing the notebook: claims and profiles contain personal and tenant information.

## 1. Setup and read-only tenant discovery

This discovers the tenant's canonical ID before any Graph writes. It does **not** prove that your account has the necessary roles. `demo_support.py` contains HTTP, diagnostic, retry, and state helpers; the important provisioning and exchange requests remain visible below.

Environment variables take precedence over `.env`. Restart the kernel after changing configuration. This notebook targets the Microsoft Entra **global cloud**.

In [ ]:
import json
import uuid
from datetime import datetime, timedelta, timezone
from html import escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

from demo_support import (
    ADMIN_PERMISSIONS, CLEANUP_PERMISSIONS, GRAPH, GRAPH_APP_ID,
    TOKEN_EXCHANGE_SCOPE, RESOURCE_PATHS, DemoState, Graph, PropagationPending,
    claim, cleanup_demo, discover_tenant, ensure_graph_grant,
    expect_directory_denial, find_service_principal, guid, recorded_resource,
    request_token, require, require_live_claims, require_unused_name, retry,
    settings_from_environment, show_claims, show_token_request, sign_in,
)

load_dotenv(dotenv_path=Path.cwd().joinpath('.env'))
settings = settings_from_environment()
TENANT_ID, AUTHORITY = retry(lambda: discover_tenant(settings['tenant']), label='Tenant discovery')
TOKEN_ENDPOINT = f'{AUTHORITY}/oauth2/v2.0/token'
PREFIX = settings['prefix']
BLUEPRINT_NAME = f'{PREFIX}-blueprint'
CLIENT_APP_NAME = f'{PREFIX}-client-app'
AGENT_NAME = f'{PREFIX}-agent'
state = DemoState(TENANT_ID, PREFIX)

print('Resolved tenant:', TENANT_ID)
print('Authority:', AUTHORITY)
print('Demo objects:', BLUEPRINT_NAME, CLIENT_APP_NAME, AGENT_NAME)
print('Private state file:', state.path)
print('Sign-in mode:', settings['auth_mode'])
print('No tenant objects have been created by this section.')

## 2. Sign in as the provisioning administrator

This is **setup**, not the three-token demonstration. These delegated Microsoft Graph permissions belong to the **bootstrap client acting with the administrator**, never to the child agent. `Application.ReadWrite.All` and `DelegatedPermissionGrant.ReadWrite.All` are powerful setup permissions.

Activate any required PIM roles before signing in. Approve the requested delegated permissions only in your chosen development tenant. The normal default bootstrap client is **Microsoft Graph Command Line Tools**, not Azure CLI. Agent ID APIs reject provisioning tokens carrying `Directory.AccessAsUser.All`.

After sign-in, the notebook checks read access and asks for confirmation before provisioning demo resources. Approving the bootstrap client during sign-in can itself create consent records; cleanup does not revoke that shared client's consent. The read checks cannot pre-prove every write permission, credential policy, quota, or Conditional Access decision. A policy denial stops the run; there is no silent switch to another identity.

In [ ]:
ADMIN_SCOPES = list(ADMIN_PERMISSIONS)
for permission, purpose in ADMIN_PERMISSIONS.items():
    print(f'{permission}: {purpose}')

admin_result = sign_in(settings['bootstrap_client_id'], AUTHORITY, ADMIN_SCOPES,
                       settings['auth_mode'], 'Administrator -- provisioning only')
admin_identity = admin_result.get('id_token_claims', {})
require(admin_identity.get('tid') == TENANT_ID, 'Administrator signed into the wrong tenant.')
granted_scopes = {scope.rsplit('/', 1)[-1] for scope in admin_result.get('scope', '').split()}
require('Directory.AccessAsUser.All' not in granted_scopes,
        'Use the documented bootstrap client with explicit scopes, not an Azure CLI token.')
require(set(ADMIN_SCOPES) <= granted_scopes,
        f'Bootstrap consent is incomplete. Missing scopes: {sorted(set(ADMIN_SCOPES) - granted_scopes)}')
admin = Graph(admin_result['access_token'])
me = admin.get('/v1.0/me', {'$select': 'id,displayName,userPrincipalName,userType'})
DEMO_USER_ID = settings['demo_user_id'] or me['id']
state.bind_user(DEMO_USER_ID)

for collection in (
    '/v1.0/applications/microsoft.graph.agentIdentityBlueprint',
    '/v1.0/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal',
    '/beta/servicePrincipals/microsoft.graph.agentIdentity',
):
    admin.get(collection, {'$top': 1, '$select': 'id'})

print(f"Provisioning administrator: {me['displayName']} ({me['userPrincipalName']})")
print('Graph consent will apply ONLY to user object ID:', DEMO_USER_ID)
print('Setup creates a blueprint, a 7-day secret, a public client, their principals and one agent identity.')
PROVISIONING_APPROVED = input(f'Type {PREFIX} to provision this demo in tenant {TENANT_ID}: ').strip() == PREFIX
require(PROVISIONING_APPROVED, 'Provisioning was not confirmed. No demo resources were created.')

## 3. Create the blueprint, its credential, API scope, and principal

The blueprint is a special **application object**. It owns the credential and exposes the API that Tc will target. Its **principal** represents it in this tenant; creating the application through Graph does not replace the separate principal-creation step.

We use typed Graph endpoints so an agent blueprint cannot be confused with an ordinary application. Saved object IDs, not just display names, identify this demo's resources. If an untracked object already has the chosen name, the notebook stops rather than adopting it.

**ID distinction:** `appId` is an OAuth client ID; `id` is a directory object ID used in Graph resource URLs.

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
blueprint = recorded_resource(admin, state, 'blueprint', RESOURCE_PATHS['blueprint'])
if blueprint is None:
    require_unused_name(admin, '/v1.0/applications/microsoft.graph.agentIdentityBlueprint', BLUEPRINT_NAME)
    blueprint = admin.post('/v1.0/applications/microsoft.graph.agentIdentityBlueprint', {
        'displayName': BLUEPRINT_NAME,
        'sponsors@odata.bind': [f"{GRAPH}/v1.0/users/{me['id']}"],
        'owners@odata.bind': [f"{GRAPH}/v1.0/users/{me['id']}"],
    })
    state.record('blueprint', blueprint)
    print('Created blueprint.')
else:
    print('Reusing the recorded blueprint.')

BLUEPRINT_OBJECT_ID = blueprint['id']
BLUEPRINT_APP_ID = blueprint['appId']
BLUEPRINT_PATH = RESOURCE_PATHS['blueprint'].format(id=BLUEPRINT_OBJECT_ID)
print('Blueprint appId (OAuth):', BLUEPRINT_APP_ID)
print('Blueprint object id (Graph):', BLUEPRINT_OBJECT_ID)

In [ ]:
ROTATE_SECRET = False  # Set True only to replace this demo's saved credential, then restore False.

require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
credential_metadata = retry(
    lambda: admin.get(BLUEPRINT_PATH, {'$select': 'id,appId,passwordCredentials'}),
    label='Read blueprint credentials', propagation=True,
)
saved_credential = state.get('blueprint_credential') or {}
live_credential = next((item for item in credential_metadata.get('passwordCredentials', [])
                        if item['keyId'] == saved_credential.get('key_id')), None)
reusable = (not ROTATE_SECRET and saved_credential.get('app_id') == BLUEPRINT_APP_ID
            and saved_credential.get('secret') and live_credential is not None
            and datetime.fromisoformat(live_credential['endDateTime'].replace('Z', '+00:00'))
            > datetime.now(timezone.utc) + timedelta(minutes=5))
if not reusable:
    credential = retry(lambda: admin.post(f'{BLUEPRINT_PATH}/addPassword', {
        'passwordCredential': {
            'displayName': f'{PREFIX}-notebook-secret',
            'endDateTime': (datetime.now(timezone.utc) + timedelta(days=7)).strftime('%Y-%m-%dT%H:%M:%SZ'),
        },
    }), label='Create blueprint credential', propagation=True)
    require(credential.get('secretText'), 'Graph returned no secretText for the new credential.')
    state.save(blueprint_credential={
        'app_id': BLUEPRINT_APP_ID, 'key_id': credential['keyId'],
        'secret': credential['secretText'], 'expires_at': credential['endDateTime'],
    })
    print('Created a development-only credential; secret value is not displayed.')
else:
    print('Reusing the saved, unexpired credential whose key still exists in Entra.')
BLUEPRINT_SECRET = state.get('blueprint_credential')['secret']
print('Secret expires:', state.get('blueprint_credential')['expires_at'])
print('Private plaintext state:', state.path)

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
blueprint = admin.get(BLUEPRINT_PATH)
api = blueprint.get('api') or {}
scopes = list(api.get('oauth2PermissionScopes') or [])
matches = [scope for scope in scopes if scope['value'] == 'access_agent']
require(len(matches) <= 1, 'Blueprint has duplicate access_agent scope definitions.')
SCOPE_ID = matches[0]['id'] if matches else str(uuid.uuid4())
agent_scope_definition = {
    'id': SCOPE_ID, 'value': 'access_agent', 'type': 'User', 'isEnabled': True,
    'adminConsentDisplayName': 'Access the agent',
    'adminConsentDescription': 'Allow the client to call the agent API for the signed-in user.',
    'userConsentDisplayName': 'Access the agent',
    'userConsentDescription': 'Allow the client to call the agent API on your behalf.',
}
scopes = [scope for scope in scopes if scope['value'] != 'access_agent'] + [agent_scope_definition]
identifier_uris = list(blueprint.get('identifierUris') or [])
if f'api://{BLUEPRINT_APP_ID}' not in identifier_uris:
    identifier_uris.append(f'api://{BLUEPRINT_APP_ID}')
admin.patch(BLUEPRINT_PATH, {
    'identifierUris': identifier_uris,
    'api': {'requestedAccessTokenVersion': 2, 'oauth2PermissionScopes': scopes},
})
AGENT_SCOPE = f'api://{BLUEPRINT_APP_ID}/access_agent'
state.save(scope_id=SCOPE_ID)
print('The frontend will request:', AGENT_SCOPE)
print('This permission is for the agent API. It grants NO Microsoft Graph access.')

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
blueprint_sp = recorded_resource(admin, state, 'blueprint_principal', RESOURCE_PATHS['blueprint_principal'])
if blueprint_sp is None:
    blueprint_sp = find_service_principal(admin, BLUEPRINT_APP_ID, blueprint=True)
    if blueprint_sp is None:
        blueprint_sp = retry(lambda: admin.post(
            '/v1.0/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal',
            {'appId': BLUEPRINT_APP_ID},
        ), label='Create blueprint principal', propagation=True)
    state.record('blueprint_principal', blueprint_sp)
require(blueprint_sp['appId'] == BLUEPRINT_APP_ID, 'Blueprint principal belongs to another application.')
print('Blueprint principal object ID:', blueprint_sp['id'])

## 4. Register the ordinary public client

A blueprint or child identity does not perform the user's interactive **token-issuing** sign-in. This ordinary client does. It is public: it has **no client secret**. MSAL uses device code, or authorization code + PKCE if `AUTH_MODE=interactive`.

`requiredResourceAccess` declares what the client requests; it is not itself consent. Preauthorizing this client on the blueprint allows `access_agent` without a separate user consent prompt for that custom scope. **This does not grant Graph `User.Read`.** That second relationship is configured in section 6.

Some Agent ID documentation also describes consent-only URLs with `response_type=none`. Those do not issue Tc and are not needed in this demo.

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
client_app = recorded_resource(admin, state, 'client_app', RESOURCE_PATHS['client_app'])
if client_app is None:
    require_unused_name(admin, '/v1.0/applications', CLIENT_APP_NAME)
    client_app = admin.post('/v1.0/applications', {
        'displayName': CLIENT_APP_NAME,
        'signInAudience': 'AzureADMyOrg',
        'isFallbackPublicClient': True,
        'publicClient': {'redirectUris': ['http://localhost']},
        'requiredResourceAccess': [{
            'resourceAppId': BLUEPRINT_APP_ID,
            'resourceAccess': [{'id': SCOPE_ID, 'type': 'Scope'}],
        }],
    })
    state.record('client_app', client_app)
CLIENT_APP_ID = client_app['appId']
CLIENT_APP_OBJECT_ID = client_app['id']

client_sp = recorded_resource(admin, state, 'client_principal', RESOURCE_PATHS['client_principal'])
if client_sp is None:
    client_sp = find_service_principal(admin, CLIENT_APP_ID)
    if client_sp is None:
        client_sp = retry(lambda: admin.post('/v1.0/servicePrincipals', {'appId': CLIENT_APP_ID}),
                          label='Create client principal', propagation=True)
    state.record('client_principal', client_sp)

api = admin.get(BLUEPRINT_PATH).get('api') or {}
preauthorized = list(api.get('preAuthorizedApplications') or [])
entry = next((item for item in preauthorized if item['appId'] == CLIENT_APP_ID), None)
if entry is None:
    preauthorized.append({'appId': CLIENT_APP_ID, 'delegatedPermissionIds': [SCOPE_ID]})
elif SCOPE_ID not in entry['delegatedPermissionIds']:
    entry['delegatedPermissionIds'].append(SCOPE_ID)
admin.patch(BLUEPRINT_PATH, {'api': {'preAuthorizedApplications': preauthorized}})
print('Public client appId:', CLIENT_APP_ID)
print('Preauthorized scope:', AGENT_SCOPE)

## 5. Create one child agent identity

The administrator creates a **real Agent ID service principal**, not another ordinary application. `agentIdentityBlueprintId` points to the blueprint's **appId**. The child has its own appId, which becomes `fmi_path` in T1 and `client_id` in the OBO request.

For a reliable, simple rehearsal, this notebook uses the administrator's delegated `AgentIdentity.Create.All` permission. A production service can also provision children using its blueprint identity. We deliberately do not add that extra app-only token flow or silently fall back between identities.

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
agent = recorded_resource(admin, state, 'agent', RESOURCE_PATHS['agent'])
if agent is None:
    require_unused_name(admin, '/beta/servicePrincipals/microsoft.graph.agentIdentity', AGENT_NAME)
    agent = retry(lambda: admin.post('/beta/servicePrincipals/microsoft.graph.agentIdentity', {
        'displayName': AGENT_NAME,
        'agentIdentityBlueprintId': BLUEPRINT_APP_ID,
        'sponsors@odata.bind': [f"{GRAPH}/v1.0/users/{me['id']}"],
    }), label='Create child agent identity', propagation=True)
    state.record('agent', agent)

def read_ready_agent():
    current = admin.get(RESOURCE_PATHS['agent'].format(id=agent['id']), {
        '$select': 'id,appId,displayName,servicePrincipalType,agentIdentityBlueprintId',
    })
    if not current.get('appId'):
        raise PropagationPending('Agent appId is not populated yet; wait and rerun this cell if retries expire.')
    return current

agent = retry(read_ready_agent, label='Read child agent', propagation=True)
require(agent.get('agentIdentityBlueprintId') == BLUEPRINT_APP_ID, 'Child has the wrong parent blueprint.')
require(agent.get('servicePrincipalType') == 'ServiceIdentity', 'Expected an Agent ID ServiceIdentity.')
state.record('agent', agent)
AGENT_ID = agent['id']
AGENT_APP_ID = agent['appId']
print('Agent appId (fmi_path and OAuth client_id):', AGENT_APP_ID)
print('Agent object ID (Graph consent clientId):', AGENT_ID)
print('Parent blueprint appId:', agent['agentIdentityBlueprintId'])

## 6. Grant Graph User.Read to this agent for one demo user

This is the authorization that permits the final exchange. The grant is:

- **Client:** the child agent's service-principal **object ID**, not its appId, blueprint, or public client.
- **Resource:** Microsoft Graph's service-principal **object ID** in this tenant.
- **Scope:** `User.Read` -- read the signed-in user's profile.
- **Consent type:** `Principal`, with the chosen demo user's object ID. **Not `AllPrincipals`** (tenant-wide consent).

We also declare Graph `User.Read` in the blueprint's `requiredResourceAccess` for transparency. The declaration is not the grant. No permission inheritance is configured by this notebook; direct grants are supported for individual agent identities.

A token does not override the user's own access or Conditional Access. In general, effective permissions can combine direct and inherited grants. For this dedicated demo, unexpected preexisting grants cause a stop instead of being silently widened or revoked.

In [ ]:
require(PROVISIONING_APPROVED, 'Confirm provisioning in section 2 first.')
graph_sp = find_service_principal(admin, GRAPH_APP_ID)
require(graph_sp is not None, 'Microsoft Graph service principal was not found in this tenant.')
GRAPH_SP_ID = graph_sp['id']
user_read_scope = next((scope for scope in graph_sp.get('oauth2PermissionScopes', [])
                        if scope['value'] == 'User.Read' and scope.get('isEnabled')), None)
require(user_read_scope is not None, 'Graph did not advertise an enabled User.Read scope.')
required_access = list(admin.get(BLUEPRINT_PATH).get('requiredResourceAccess') or [])
graph_access = next((item for item in required_access if item['resourceAppId'] == GRAPH_APP_ID), None)
if graph_access is None:
    graph_access = {'resourceAppId': GRAPH_APP_ID, 'resourceAccess': []}
    required_access.append(graph_access)
user_read_reference = {'id': user_read_scope['id'], 'type': 'Scope'}
if user_read_reference not in graph_access['resourceAccess']:
    graph_access['resourceAccess'].append(user_read_reference)
admin.patch(BLUEPRINT_PATH, {'requiredResourceAccess': required_access})

grant = ensure_graph_grant(admin, state, AGENT_ID, GRAPH_SP_ID, DEMO_USER_ID)
print('Live consent record (IDs are directory object IDs):')
print(json.dumps({key: grant.get(key) for key in
                  ('id', 'clientId', 'resourceId', 'consentType', 'principalId', 'scope')}, indent=2))
print('Provisioning is complete. The audience walkthrough starts at section 7.')

## 7. Tc: the user signs in to the public client

**Say to the audience:** "First, the user gives the agent a ticket addressed to the agent's API. This is not a pass to Graph."

Sign in as the **demo user selected in section 2**. The public client asks for `api://<blueprint-appId>/access_agent`. MSAL handles the device-code protocol (or browser authorization code + PKCE) and returns an **access token**. Tc is not the ID token also returned during sign-in.

Watch **`aud`** (blueprint), **`scp`** (`access_agent`), **`oid`** (user), and **`azp`** (public client). The checks below are diagnostic consistency checks, not a replacement for signature validation by a production API.

In [ ]:
print('Use demo user object ID:', DEMO_USER_ID)
tc_result = sign_in(CLIENT_APP_ID, AUTHORITY, [AGENT_SCOPE], settings['auth_mode'], 'Tc -- user sign-in')
Tc = tc_result['access_token']
tc_claims = show_claims(Tc, ['aud', 'scp', 'oid', 'azp|appid', 'tid', 'preferred_username|upn', 'exp'],
                        'Tc: user access token FOR the blueprint API')
require_live_claims(tc_claims, 'section 7')
require(tc_claims.get('aud') in (BLUEPRINT_APP_ID, f'api://{BLUEPRINT_APP_ID}'),
        'Tc must target the blueprint API, not Microsoft Graph.')
require(tc_claims.get('tid') == TENANT_ID, 'Tc is from the wrong tenant.')
require(tc_claims.get('oid') == DEMO_USER_ID, 'Wrong user. Rerun section 7 as the user selected in section 2.')
require(claim(tc_claims, 'azp', 'appid') == CLIENT_APP_ID, 'Tc was not issued to this public client.')
require('access_agent' in tc_claims.get('scp', '').split(), 'Tc does not carry access_agent.')
print('Tc has the expected audience, user, tenant, client and API scope.')

## 8. T1: the blueprint obtains a credential for its child agent

**Say:** "Now the blueprint proves who it is and says which agent it represents. Entra gives it an exchange token -- the agent's badge. No user token is used in this request."

`client_id` is the **blueprint**. `fmi_path` is the **child's appId**. `api://AzureADTokenExchange/.default` targets the token-exchange resource. T1 is an access token that will be reused as a **client assertion**, not sent to Graph as the final user token.

Watch `aud`, `azp`/`appid`, and `sub`. Entra binds the FMI subject to the child. **Do not assume that `sub` equals, or literally contains, the child's appId**; its serialization is not a parsing contract for this notebook.

In [ ]:
t1_form = {
    'grant_type': 'client_credentials',
    'client_id': BLUEPRINT_APP_ID,
    'client_secret': BLUEPRINT_SECRET,
    'scope': TOKEN_EXCHANGE_SCOPE,
    'fmi_path': AGENT_APP_ID,
}
show_token_request(TOKEN_ENDPOINT, t1_form, title='T1 request -- blueprint proves the child identity')
t1_result = retry(lambda: request_token(TOKEN_ENDPOINT, **t1_form),
                  label='T1 exchange token', propagation=True)
T1 = t1_result['access_token']
t1_claims = show_claims(T1, ['aud', 'azp|appid', 'sub', 'oid', 'idtyp', 'tid', 'exp'],
                        'T1: exchange token used as the agent credential')
require_live_claims(t1_claims, 'section 8')
require(claim(t1_claims, 'azp', 'appid') == BLUEPRINT_APP_ID, 'T1 must be obtained by this blueprint.')
require(t1_claims.get('tid') == TENANT_ID, 'T1 is from the wrong tenant.')
print('The next request reuses this exact T1 fingerprint as client_assertion.')

## 9. Tg: the child exchanges the badge and ticket for a Graph token

**Say:** "The agent presents two things: T1 proves the agent identity; Tc represents the signed-in user. Entra checks both, plus Graph consent and policy, before issuing the Graph pass."

**Notice the switch:** `client_id` is now the **child agent's appId**, not the blueprint's. `client_assertion=T1` authenticates the child; `assertion=Tc` is the user's incoming access token. These two assertions have different jobs.

`requested_token_use=on_behalf_of` and the JWT-bearer grant request OBO. Here we use `https://graph.microsoft.com/.default` to request the already-granted Graph permissions. `/.default` does not mean "every Graph permission" and cannot create consent.

In a readable Tg, `aud` changes to Graph, `oid` remains the same user, and `azp`/`appid` changes to the agent. Optional agent facet claims are displayed when present. Graph owns this token format: an opaque Graph token can still work.

In [ ]:
require_live_claims(tc_claims, 'section 7, then sections 8-9')
require_live_claims(t1_claims, 'section 8, then section 9')
tg_form = {
    'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
    'client_id': AGENT_APP_ID,
    'client_assertion_type': 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer',
    'client_assertion': T1,
    'assertion': Tc,
    'requested_token_use': 'on_behalf_of',
    'scope': f'{GRAPH}/.default',
}
show_token_request(TOKEN_ENDPOINT, tg_form, title='Tg request -- the actual OBO exchange',
                   token_names={'client_assertion': 'T1', 'assertion': 'Tc'})
tg_result = retry(lambda: request_token(TOKEN_ENDPOINT, **tg_form),
                  label='Graph OBO token', propagation=True, consent_pending=True)
Tg = tg_result['access_token']
tg_claims = show_claims(Tg, ['aud', 'scp', 'oid', 'azp|appid', 'tid', 'idtyp',
                           'xms_act_fct', 'xms_sub_fct', 'exp'], 'Tg: Graph access token FOR the user VIA the agent')
if tg_claims is not None:
    require(tg_claims.get('aud') in (GRAPH_APP_ID, GRAPH, GRAPH + '/'), 'Tg must target Microsoft Graph.')
    require(tg_claims.get('oid') == tc_claims['oid'], 'Tg must represent the same user as Tc.')
    require(tg_claims.get('tid') == TENANT_ID, 'Tg is from the wrong tenant.')
    require(claim(tg_claims, 'azp', 'appid') == AGENT_APP_ID, 'The actor in Tg must be the child agent.')
scope_text = tg_claims.get('scp') if tg_claims is not None else tg_result.get('scope')
if scope_text is not None:
    graph_scopes = set(scope_text.split()) - {'openid', 'profile', 'email', 'offline_access'}
    require(graph_scopes == {'User.Read'}, f'Unexpected effective Graph scopes: {sorted(graph_scopes)}')
    print('Effective Graph scope: User.Read.')
else:
    print('Effective scopes are not inspectable here; the consent record and Graph calls are shown instead.')
print('Entra issued Tg. Now use it -- without the administrator token -- to call Graph.')

## 10. Use Tg: one operation allowed, one denied

**Say:** "The pass works for this user's profile, but it is not permission to browse the directory."

`GET /me` needs delegated `User.Read`. `GET /users` needs at least delegated `User.ReadBasic.All`, which this demo does not grant. This comparison does not depend on an Exchange mailbox or mail license. Use a **member user** so guest-user restrictions do not confound the example.

Only a recognized **HTTP 403 access-denied** response counts as the expected negative result. An expired token, network problem, HTTP 400/401/404/500, or unexpected success is not a passing demonstration.

Optional: in the Entra admin center, open **Entra ID > Monitoring & health > Sign-in logs**, and filter **Is Agent = Yes** / **Agent type = Agent Identity**. Viewing logs requires a role such as **Reports Reader**. Allow time for ingestion. Sign-in logs describe authentication/token activity, not an audit of each `GET /me` request.

In [ ]:
as_user_via_agent = Graph(Tg)
profile = as_user_via_agent.get('/v1.0/me', {
    '$select': 'id,displayName,userPrincipalName,jobTitle,userType',
})
require(profile.get('id') == DEMO_USER_ID, 'Graph returned a different user from the one who supplied Tc.')
require(profile.get('userType') == 'Member', 'Use a member user for the directory-permission comparison.')
print('GET /me succeeded using Tg:')
for key in ('displayName', 'userPrincipalName', 'id', 'jobTitle'):
    print(f'  {key}: {profile.get(key)}')
print('\nNow GET /users with the SAME Tg, without granting User.ReadBasic.All:')
expect_directory_denial(as_user_via_agent)
print('\nLive demonstration complete: same user, identified agent, resource-specific access.')

## 11. Recap: follow the audience, user, and actor

**Tc:** frontend + user sign-in, addressed to the blueprint.  
**T1:** blueprint credential + child `fmi_path`, addressed to the token-exchange resource.  
**Tg:** child identity + `client_assertion=T1` + `assertion=Tc`, addressed to Graph.

The user is not replaced by an application-only identity. The user stays the subject while the actor changes from frontend to child agent. `access_agent` does not turn into `User.Read` by renaming a scope: Entra issues a **new token for a different API** after checking the separate Graph grant.

If you pause long enough for tokens to expire, rerun sections 7-10. Keep setup out of the live presentation whenever possible. See the README for accurate error-code meanings and tenant-policy troubleshooting.

In [ ]:
aliases = {BLUEPRINT_APP_ID: 'blueprint', CLIENT_APP_ID: 'public client',
           AGENT_APP_ID: 'child agent', GRAPH_APP_ID: 'Microsoft Graph', GRAPH: 'Microsoft Graph'}

def table_value(claims, key):
    if claims is None:
        return 'opaque token -- not decoded'
    value = claim(claims, *key.split('|'))
    if value is None:
        return 'not present'
    text = json.dumps(value, ensure_ascii=True) if isinstance(value, (dict, list)) else str(value)
    if text in aliases:
        text = f'{aliases[text]} ({text})'
    return escape(text).replace('|', '&#124;').replace('`', '&#96;').replace('\n', ' ')

rows = ['| Claim | Tc | T1 | Tg |', '|---|---|---|---|']
for key in ('aud', 'azp|appid', 'oid', 'scp', 'sub', 'xms_act_fct', 'xms_sub_fct'):
    values = [table_value(claims, key) for claims in (tc_claims, t1_claims, tg_claims)]
    rows.append('| ' + key.replace('|', '/') + ' | ' + ' | '.join(values) + ' |')
display(Markdown('\n'.join(rows)))

## 12. Optional cleanup -- leave off during the presentation

Set `CLEANUP = True`, sign in with the cleanup permissions, and confirm the prefix. You can run **section 1 and this section** to recover from a partial setup or partial cleanup; completed provisioning is not required.

Cleanup explicitly removes the recorded grant, child agent, public-client principal/application, and blueprint principal/application. It checks recorded IDs and refuses a blueprint with additional child identities. It does not rely on an undocumented cascade to find everything.

**Any failure keeps the state file**, including its credential, so you can resolve permissions and rerun cleanup. The three extra delegated permissions are `AgentIdentity.DeleteRestore.All`, `AgentIdentityBlueprintPrincipal.DeleteRestore.All`, and `AgentIdentityBlueprint.DeleteRestore.All`; the ordinary client uses `Application.ReadWrite.All`.

Agent objects are soft-deleted for 30 days. Recovery of Agent ID objects uses Graph or Microsoft Entra PowerShell, not an assumed admin-center restore button. Deleting the local state file alone does not remove tenant objects or revoke the blueprint credential. Already-issued access tokens can remain usable until expiration.

**Before sharing:** clear all notebook outputs, save, and restart the kernel to discard in-memory credentials. Never share the private state file or `.env`.

In [ ]:
CLEANUP = False

if CLEANUP:
    print('Cleanup tenant:', TENANT_ID)
    print('Cleanup prefix:', PREFIX)
    print('Cleanup state:', state.path)
    require(input(f'Type delete {PREFIX} to remove the recorded demo objects: ').strip() == f'delete {PREFIX}',
            'Cleanup was not confirmed. Nothing deleted.')
    cleanup_result = sign_in(settings['bootstrap_client_id'], AUTHORITY, CLEANUP_PERMISSIONS,
                             settings['auth_mode'], 'Administrator -- cleanup only')
    require(cleanup_result.get('id_token_claims', {}).get('tid') == TENANT_ID,
            'Cleanup administrator signed into the wrong tenant.')
    cleanup_demo(Graph(cleanup_result['access_token']), state)
else:
    print('Nothing deleted. Leave CLEANUP=False while rehearsing or presenting.')